In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sellers_path = "sellers_final_20260331_232908.csv"
geolocation_path = "geolocation_final_20260331_233312.csv"

sellers_df = spark.read.csv(sellers_path, header=True, inferSchema=True)
geolocation_df = spark.read.csv(geolocation_path, header=True, inferSchema=True)

# Create an index per zip for both tables and map sellers to geolocations by modulo
geo_w = Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.monotonically_increasing_id())
geo_indexed = (
    geolocation_df
    .withColumn("geo_index", F.row_number().over(geo_w))
    .withColumn("geo_count", F.count("*").over(Window.partitionBy("geolocation_zip_code_prefix")))
    .withColumnRenamed("geolocation_zip_code_prefix", "seller_zip_code_prefix")
)

geo_counts = geo_indexed.select("seller_zip_code_prefix", "geo_count").dropDuplicates(["seller_zip_code_prefix"])

seller_w = Window.partitionBy("seller_zip_code_prefix").orderBy(F.monotonically_increasing_id())
sellers_indexed = sellers_df.withColumn("seller_index", F.row_number().over(seller_w))

sellers_with_geo_count = sellers_indexed.join(
    geo_counts,
    on="seller_zip_code_prefix",
    how="inner"
)

sellers_with_geo_index = sellers_with_geo_count.withColumn(
    "geo_index",
    F.pmod(F.col("seller_index") - F.lit(1), F.col("geo_count")) + F.lit(1)
)

joined_df = sellers_with_geo_index.join(
    geo_indexed,
    on=["seller_zip_code_prefix", "geo_index"],
    how="inner"
)

joined_df = joined_df.drop("geo_count", "geo_index", "seller_index")

joined_df.show(5)
print(f"Total rows: {joined_df.count()}")

+----------------------+--------------------+-----------+-------------------+-------------------+
|seller_zip_code_prefix|           seller_id|seller_city|    geolocation_lat|    geolocation_lng|
+----------------------+--------------------+-----------+-------------------+-------------------+
|                  1031|d594982fd877af63a...|  SAO PAULO|-23.536864121011018| -46.63349313820771|
|                  1031|f4aba7c0bca51484c...|  SAO PAULO|-23.537304315614186|-46.633862438491114|
|                  1127|f5b84683a9bf9e1df...|  SAO PAULO|-23.527735040250406| -46.64480016028046|
|                  1139|1f7dfad2cb384ea4d...|  SAO PAULO| -23.52076037607479|-46.661384138207694|
|                  1156|74a9b9bddf14ece02...|  SAO PAULO|  -23.5298272739339| -46.66466605744709|
+----------------------+--------------------+-----------+-------------------+-------------------+
only showing top 5 rows

Total rows: 1814


In [4]:
from pyspark.sql import functions as F
import csv
import os
import pandas as pd
from datetime import datetime

joined_pd = joined_df.toPandas()
output_dir = os.getcwd()

output_path = os.path.join(
    output_dir, f"sellers_and_geolocation_final_{datetime.now():%Y%m%d_%H%M%S}.csv"
 )
joined_pd.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

CSV salvo em: c:\Users\Sofhia\Downloads\archive\sellers_and_geolocation_final_20260403_140220.csv
